In [0]:
%run ./01_setup_volumen

In [0]:
import polars as pl
from datetime import datetime, timedelta, timezone
import random
import json

random.seed(42)

countries = ["ES", "MX", "CO", "AR", "CL"]
categories = ["electronics", "fashion", "home", "sports", "books"]
payment_methods = ["card", "transfer", "wallet"]

# customers.csv
customers = pl.DataFrame({
    "customer_id": list(range(1, settings.n_customers + 1)),
    "name": [f"customer_{i}" for i in range(1, settings.n_customers + 1)],
    "email": [f"customer_{i}@lab.dev" for i in range(1, settings.n_customers + 1)],
    "country": [random.choice(countries) for _ in range(settings.n_customers)],
    "signup_date": [
        (datetime(2025, 1, 1) + timedelta(days=random.randint(0, 365))).date().isoformat()
        for _ in range(settings.n_customers)
    ],
})

print(settings.raw)
customers.write_csv(settings.raw / "customers.csv")

In [0]:
customers.head()

In [0]:
# products.parquet
products = pl.DataFrame({
    "product_id": list(range(1, settings.n_products + 1)),
    "product_name": [f"product_{i}" for i in range(1, settings.n_products + 1)],
    "category": [random.choice(categories) for _ in range(settings.n_products)],
    "unit_price": [round(random.uniform(5, 500), 2) for _ in range(settings.n_products)],
    "active": [random.random() > 0.03 for _ in range(settings.n_products)],
})
products.write_parquet(settings.raw / "products.parquet")

In [0]:
products.head()

In [0]:
price_map = dict(zip(products["product_id"].to_list(), products["unit_price"].to_list()))

# orders.jsonl y payments.csv
statuses = ["paid", "cancelled", "pending"]
weights = [0.78, 0.10, 0.12]
base_ts = datetime(2026, 3, 1, tzinfo=timezone.utc)

orders_records = []
payments_records = []
payment_id = 1

for order_id in range(1, settings.n_orders + 1):
    customer_id = random.randint(1, settings.n_customers)
    product_id = random.randint(1, settings.n_products)
    quantity = random.randint(1, 5)
    status = random.choices(statuses, weights=weights, k=1)[0]
    order_ts = base_ts + timedelta(minutes=random.randint(0, 60 * 24 * 28))

    orders_records.append({
        "order_id": order_id,
        "customer_id": customer_id,
        "product_id": product_id,
        "quantity": quantity,
        "order_ts": order_ts.isoformat(),
        "status": status,
    })

    if status == "paid":
        amount = round(quantity * float(price_map[product_id]), 2)
        payments_records.append({
            "payment_id": payment_id,
            "order_id": order_id,
            "amount": amount,
            "currency": "EUR",
            "payment_method": random.choice(payment_methods),
            "paid_at": (order_ts + timedelta(minutes=random.randint(1, 30))).isoformat(),
        })
        payment_id += 1

with open(settings.raw / "orders.jsonl", "w", encoding="utf-8") as f:
    for rec in orders_records:
        f.write(json.dumps(rec) + "\n")

pl.DataFrame(payments_records).write_csv(settings.raw / "payments.csv")


In [0]:
orders= pl.DataFrame(orders_records)
orders.shape


In [0]:
print("Archivos fuente creados en:", settings.raw)
print((settings.raw / "customers.csv").exists())
print((settings.raw / "products.parquet").exists())
print((settings.raw / "orders.jsonl").exists())
print((settings.raw / "payments.csv").exists())